In [1]:
# Cellule 1 : imports
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score, roc_curve

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

import matplotlib.pyplot as plt

In [2]:
# Cellule 2 : chargement du CSV
df = pd.read_csv("data/GermanCredit.csv")
df.head()
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 7 columns):
 #   Column                 Non-Null Count  Dtype
---  ------                 --------------  -----
 0   Duration               1000 non-null   int64
 1   Amount                 1000 non-null   int64
 2   Age                    1000 non-null   int64
 3   CheckingAccountStatus  1000 non-null   str  
 4   SavingsAccount         1000 non-null   str  
 5   Purpose                1000 non-null   str  
 6   Class                  1000 non-null   str  
dtypes: int64(3), str(4)
memory usage: 54.8 KB


In [3]:
# Cellule 3 : séparation X / y
# On garde la même convention : Class = cible
y = df["Class"].copy()
X = df.drop(columns=["Class"]).copy()

# Encodage de la cible en 0/1 (Good=0, Bad=1 par ex.)
y = (y == "Bad").astype(int)  # Bad -> 1, Good -> 0
y.value_counts()

Class
0    700
1    300
Name: count, dtype: int64

In [4]:
# Cellule 4 : gérer les variables catégorielles
cat_cols = ["CheckingAccountStatus", "SavingsAccount", "Purpose"]
num_cols = ["Duration", "Amount", "Age"]

# One-hot encoding pour les qualitatives
X_cat = pd.get_dummies(X[cat_cols], drop_first=False)

# On garde les colonnes numériques à part
X_num = X[num_cols]

# On concatène
X_all = pd.concat([X_num, X_cat], axis=1)
X_all.head()

,Duration,Amount,Age,CheckingAccountStatus_0-200,CheckingAccountStatus_<0,CheckingAccountStatus_>200,SavingsAccount_100-500,SavingsAccount_500-1000,SavingsAccount_<100,SavingsAccount_>1000,Purpose_NewCar,Purpose_Other,Purpose_UsedCar
0,6,1169,67,False,True,False,False,False,False,True,False,True,False
1,48,5951,22,True,False,False,False,False,True,False,False,True,False
2,12,2096,49,False,False,True,False,False,True,False,False,True,False
3,42,7882,45,False,True,False,False,False,True,False,False,True,False
4,24,4870,53,False,True,False,False,False,True,False,True,False,False


In [5]:
# Cellule 5 : split train / test (stratifié sur la classe)
X_train, X_test, y_train, y_test = train_test_split(
    X_all.values,
    y.values,
    test_size=0.3,
    random_state=123,
    stratify=y.values
)

X_train.shape, X_test.shape

((700, 13), (300, 13))

In [6]:
# Cellule 6 : standardisation des variables numériques
scaler = StandardScaler()

# On standardise SEULEMENT à partir du train
X_train_num = scaler.fit_transform(X_train[:, :len(num_cols)])
X_test_num  = scaler.transform(X_test[:, :len(num_cols)])

# Les colonnes catégorielles (one-hot) ne sont pas standardisées
X_train_cat = X_train[:, len(num_cols):]
X_test_cat  = X_test[:, len(num_cols):]

# On reconcatène num + cat
X_train_scaled = np.concatenate([X_train_num, X_train_cat], axis=1)
X_test_scaled  = np.concatenate([X_test_num,  X_test_cat], axis=1)

X_train_scaled.shape, X_test_scaled.shape

#X_train_scaled[:5]
#X_train_scaled.dtype

#X_train_scaled

((700, 13), (300, 13))

In [7]:
# Cellule 7 : conversion en tenseurs

X_train_scaled = X_train_scaled.astype(np.float32)
X_test_scaled  = X_test_scaled.astype(np.float32)

X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
X_test_tensor  = torch.tensor(X_test_scaled,  dtype=torch.float32)

y_train_tensor = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)  # shape (N,1)
y_test_tensor  = torch.tensor(y_test,  dtype=torch.float32).unsqueeze(1)

In [8]:
# Cellule 8 : DataLoader
batch_size = 32

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset  = TensorDataset(X_test_tensor,  y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=batch_size, shuffle=False)


In [9]:
input_dim = X_train_scaled.shape[1]

class CreditNetImproved(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(32, 1)  # logit
        )

    def forward(self, x):
        return self.net(x)

model = CreditNetImproved(input_dim)

In [10]:
# Cellule 10 : définition de la loss et de l'optimiseur
# y_train est un vecteur 0/1 (Bad=1)
pos_weight_value = (y_train == 0).sum() / (y_train == 1).sum()
pos_weight = torch.tensor([pos_weight_value], dtype=torch.float32)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)

In [11]:
# Cellule 11 : boucle d'entraînement
num_epochs = 100

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        outputs = model(X_batch)           # logits
        loss = criterion(outputs, y_batch) # y_batch en float
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * X_batch.size(0)
    
    epoch_loss = running_loss / len(train_loader.dataset)
    
    if (epoch+1) % 10 == 0:
        print(f"Epoch {epoch+1}/{num_epochs}, Loss: {epoch_loss:.4f}")


Epoch 10/100, Loss: 0.7999


Epoch 20/100, Loss: 0.7704


Epoch 30/100, Loss: 0.7507


Epoch 40/100, Loss: 0.7521


Epoch 50/100, Loss: 0.7590


Epoch 60/100, Loss: 0.7425


Epoch 70/100, Loss: 0.7464


Epoch 80/100, Loss: 0.6970


Epoch 90/100, Loss: 0.7233


Epoch 100/100, Loss: 0.7262


In [12]:
# Cellule 12 : prédictions sur le test
model.eval()
with torch.no_grad():
    logits_test = model(X_test_tensor)              # shape (N,1)
    prob_test = torch.sigmoid(logits_test).numpy()  # proba d'être Bad (classe 1)

prob_test[:5]

array([[0.81136173],
       [0.01886779],
       [0.45910567],
       [0.2706087 ],
       [0.3263751 ]], dtype=float32)

In [13]:
# Cellule 13 : seuil de décision
threshold = 0.4 # tu peux mettre 0.35 / 0.4 etc.
y_pred_test = (prob_test >= threshold).astype(int)

# Matrice de confusion
cm = confusion_matrix(y_test, y_pred_test)
cm

array([[137,  73],
       [ 25,  65]])

In [14]:
# Cellule 14 : rapport détaillé
print(classification_report(y_test, y_pred_test, target_names=["Good","Bad"]))

              precision    recall  f1-score   support

        Good       0.85      0.65      0.74       210
         Bad       0.47      0.72      0.57        90

    accuracy                           0.67       300
   macro avg       0.66      0.69      0.65       300
weighted avg       0.73      0.67      0.69       300

